# exp-040 — query masking (raw / uniform / typed) on the v2 pipeline (Colab)

Re-runs the v1 masking comparison on the current fixed method: Mistral-7B-Instruct-v0.3 at 4-bit,
unguided, the size2000_imbalanced pool, e5-small-v2 top-20 → cross-encoder top-5, the pinned 600,
exp-020's decoding. **Only the retrieval mode moves** (which masked form of the question the
retriever matches on); the prompt always shows raw questions. Design: `experiments/exp-040/README.md`,
`experiments/exp-040/config.json`, ADR 0046.

**Before you run**
1. Runtime → **A100** (any GPU with ≥ 12 GiB works; estimates are for an A100).
2. Colab secret `HF_TOKEN` (account has accepted the Mistral-7B-Instruct-v0.3 licence), or `HF_TOKEN` in the environment.
3. `MyDrive/thesis-backup.tar.gz` exists. The notebook clones the public repo; the first launch pins the commit
   (`MyDrive/thesis-exp-040/launch_sha.txt`).

**Steps:** setup → rebuild the all-modes retrieval artifact on CPU (sha256 pinned on Drive at first build) →
for each arm a 20-question canary, then the full 600 (typed_rerun = exp-020's exact command, raw, uniform) →
typed_rebuilt only if the rebuild's typed exemplars differ from exp-014's → paired compare + Holm.
Estimate (not measured): **~2.3–3.4 h on an A100**, +0.55–0.85 h if typed_rebuilt runs.

**If the runtime dies:** run all cells again. Outputs live on Drive; finished steps are skipped and an interrupted
decode resumes (`run_decomposer.py --resume`, which refuses if anything that decides the output changed).

In [ ]:
# ---- settings ----
REPO_URL = "https://github.com/AhmadiJahid/Thesis---QAv2.git"
BRANCH = "main"
COMMIT = ""                                  # "" = pin the branch head at first launch; or a full 40-hex SHA
ARCHIVE_ON_DRIVE = "/content/drive/MyDrive/thesis-backup.tar.gz"
ARCHIVE_SHA256 = "33c6072f0e4e46ab94f40f804463c512f439823b80c3e0e9dec0ee1023717d94"   # as in exp-037
DRIVE_OUT = "/content/drive/MyDrive/thesis-exp-040"
WORK = "/content/exp040"                     # clone: WORK/Thesis---QAv2, data: WORK/thesis-qav2-data
VENV = "/content/venv313_exp040"
PY = f"{VENV}/bin/python"
ARMS = ["typed_rerun", "raw", "uniform"]     # all three required (experiments/exp-040/config.json)

In [ ]:
# ---- Drive (skipped when already mounted, e.g. a runner executing this notebook with nbconvert) ----
import os, json, hashlib, subprocess, shutil, time, datetime, glob, re, threading
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(f"{DRIVE_OUT}/logs", exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/runs/exp-040", exist_ok=True)
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(ARCHIVE_ON_DRIVE), f"archive not found: {ARCHIVE_ON_DRIVE}"

def sh(cmd, log=None, env=None, cwd=None, check=True):
    # Run a shell command, stream its output here and (optionally) append it to a Drive log.
    print(f"$ {cmd}", flush=True)
    full_env = {**os.environ, **(env or {})}
    with subprocess.Popen(cmd, shell=True, cwd=cwd, env=full_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, executable="/bin/bash") as p:
        fh = open(log, "a") if log else None
        if fh: fh.write(f"\n### {datetime.datetime.now().isoformat()} $ {cmd}\n")
        for line in p.stdout:
            if "Processed" not in line:
                print(line, end="", flush=True)
            if fh: fh.write(line); fh.flush()
        rc = p.wait()
        if fh: fh.write(f"### rc={rc}\n"); fh.close()
    if check and rc != 0:
        raise RuntimeError(f"command failed (rc={rc}): {cmd}")
    return rc

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
print("Drive OK; outputs ->", DRIVE_OUT)

In [ ]:
# ---- clone (public repo); the FIRST launch pins its commit on Drive, every later launch checks it out ----
REPO = f"{WORK}/Thesis---QAv2"
if not os.path.isdir(f"{REPO}/.git"):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch -q origin {BRANCH}")
PIN_FILE = f"{DRIVE_OUT}/launch_sha.txt"
pinned = open(PIN_FILE).read().strip() if os.path.exists(PIN_FILE) else None
if COMMIT and pinned and COMMIT != pinned:
    raise SystemExit(f"COMMIT={COMMIT} differs from the pinned {pinned} ({PIN_FILE}); finished steps on Drive "
                     "came from the pinned commit - use it, or start a fresh DRIVE_OUT")
SHA = COMMIT or pinned or subprocess.check_output(["git", "-C", REPO, "rev-parse", f"origin/{BRANCH}"], text=True).strip()
assert re.fullmatch(r"[0-9a-f]{40}", SHA), SHA
sh(f"git -C {REPO} checkout -q --detach {SHA}")
assert subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip() == SHA
if not pinned:
    open(PIN_FILE, "w").write(SHA + "\n")
assert not subprocess.check_output(["git", "-C", REPO, "status", "--porcelain", "--untracked-files=no"],
                                   text=True).strip(), "clone is not clean"
os.chdir(REPO)
print("pinned commit:", SHA)

In [ ]:
# ---- GPU gate (total VRAM; another job may share the GPU - timings in the README assume it does not) ----
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,compute_cap,driver_version",
                             "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
name, mem_mib, used_mib, cc, driver = [x.strip() for x in q.split(",")]
print(f"GPU {name}: {int(mem_mib)/1024:.1f} GiB total, {int(used_mib)/1024:.1f} GiB in use, sm_{cc}, driver {driver}")
assert int(mem_mib) / 1024 >= 12, "need >= 12 GiB VRAM: Runtime -> Change runtime type -> A100"
if "A100" not in name:
    print("NOTE: not an A100 - the README's time estimates are for an A100")
GPU_INFO = {"name": name, "memory_total_mib": int(mem_mib), "driver": driver, "compute_cap": cc}
open(f"{DRIVE_OUT}/logs/gpu.txt", "a").write(f"{datetime.datetime.now().isoformat()} {q}\n")

In [ ]:
# ---- Python 3.13.9 venv with the repo's pins (the exp-026 environment, docs/compute.md "Colab") ----
if not os.path.exists(f"{VENV}/.exp040_ok"):
    sh("pip -q install uv")
    sh(f"uv venv --python 3.13.9 {VENV}")
    sh(f"uv pip install --python {PY} torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128")
    sh(f"uv pip install --python {PY} -r {REPO}/requirements.txt")
    open(f"{VENV}/.exp040_ok", "w").write("ok")
ENV_JSON = f"{DRIVE_OUT}/logs/environment.json"
sh(f'''{PY} - <<'EOF' > {ENV_JSON}
import json, sys, torch, transformers, peft, trl, bitsandbytes, accelerate, datasets, huggingface_hub, sentence_transformers
print(json.dumps({{"python": sys.version.split()[0], "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
  "cuda_available": torch.cuda.is_available(), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None,
  "transformers": transformers.__version__, "peft": peft.__version__, "trl": trl.__version__,
  "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__, "datasets": datasets.__version__,
  "huggingface_hub": huggingface_hub.__version__, "sentence_transformers": sentence_transformers.__version__}}, indent=2))
EOF''')
env_rec = json.load(open(ENV_JSON)); print(json.dumps(env_rec, indent=1))
want = {"python": "3.13.9", "torch": "2.10.0", "transformers": "5.5.4", "peft": "0.20.0", "trl": "1.10.0",
        "bitsandbytes": "0.49.2", "accelerate": "1.12.0", "datasets": "4.8.4", "huggingface_hub": "1.11.0",
        "sentence_transformers": "5.2.3"}
off = {k: (env_rec[k], v) for k, v in want.items() if not str(env_rec[k]).startswith(v)}
assert env_rec["cuda_available"], "CUDA not available inside the venv"
assert not off, f"version drift from requirements.txt: {off}"
os.environ["PY"] = PY
print("environment matches requirements.txt")

In [ ]:
# ---- Hugging Face token (Mistral-7B-Instruct-v0.3 is gated): environment first, then Colab secrets ----
if not os.environ.get("HF_TOKEN"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
sh(f'{PY} -c "from huggingface_hub import whoami; print(\'HF user:\', whoami()[\'name\'])"')

In [ ]:
# ---- data: private copy of the archive (atomic, so a concurrent notebook never reads a half copy),
#      then scripts/colab_setup_data.py extracts only what exp-040 reads (safe extraction, sha-checked) ----
LOCAL_TAR = f"/content/thesis-backup.exp-040.tar.gz"
if not os.path.exists(LOCAL_TAR):
    t = time.time(); shutil.copy(ARCHIVE_ON_DRIVE, LOCAL_TAR + ".part"); os.replace(LOCAL_TAR + ".part", LOCAL_TAR)
    print(f"copied archive in {time.time()-t:.0f}s")
sh(f"{PY} scripts/colab_setup_data.py --archive {LOCAL_TAR} --sha256 {ARCHIVE_SHA256} --parts musique "
   "--runs pool_sweep/pools/size2000_imbalanced_trial0_poolseed42/pool.jsonl exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl exp-032/exp-020/eval exp-032/exp-020/reparse", log=f"{DRIVE_OUT}/logs/setup_data.log")
# runs/exp-040 IS the Drive folder: every finished row (results.partial.jsonl) is on Drive at once.
link = "runs/exp-040"
if not os.path.islink(link):
    assert not os.path.exists(link), f"{link} exists and is not the Drive link"
    os.symlink(f"{DRIVE_OUT}/runs/exp-040", link)
print(link, "->", os.readlink(link))

In [ ]:
# ---- step runner: experiments/exp-040/run.sh <step>, log streamed to Drive, VRAM sampled ----
def run_step(*step):
    tag = "_".join(step)
    log_path = f"{DRIVE_OUT}/logs/{tag}.log"
    peak = {"mib": 0}; stop = threading.Event()
    def sampler():
        while not stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True).stdout.strip()
            if out: peak["mib"] = max(peak["mib"], int(out.splitlines()[0]))
            stop.wait(10)
    th = threading.Thread(target=sampler, daemon=True); th.start()
    started = time.time()
    rc = sh("bash experiments/exp-040/run.sh " + " ".join(step), log=log_path, env={"PY": PY}, check=False)
    stop.set(); th.join()
    rec = {"step": tag, "rc": rc, "wall_seconds": round(time.time() - started),
           "peak_vram_mib_sampled_whole_gpu": peak["mib"], "gpu": GPU_INFO, "git_sha": SHA,
           "finished": datetime.datetime.now().isoformat()}
    with open("runs/exp-040/colab_steps.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(rec)
    assert rc == 0, f"{tag} failed (rc={rc}; rc=3 is a failed output-health gate); see {log_path}. " \
                    "Do not re-run with changed settings: a changed run is a new experiment."

## 1. Retrieval artifact (CPU)
exp-014's two commands with `--mode all` and `--no-cache`; checked (600 ids = exp-014's, 200/hop, ≥ 5 candidates per mode)
and its sha256 pinned on Drive. Reports how many queries get exactly exp-014's typed exemplars.

In [ ]:
run_step("retrieval")
CHECK = json.load(open("runs/exp-040/retrieval/check.json"))
ti = CHECK["typed_identity"]
print("artifact sha256:", CHECK["artifact_sha256"])
print(f"typed exemplars identical to exp-014 (ordered): {ti['identical']}/{ti['compared']}; as a set: {ti['identical_as_set_any_order']}")
print("mode overlap (top-5):", json.dumps(CHECK["mode_overlap_top_k"], indent=1))
NEED_TYPED_REBUILT = ti["identical"] != 600
print("typed_rebuilt required:", NEED_TYPED_REBUILT)

## 2. Arms: 20-question canary, then the full 600
Each canary must pass the output-health gate before its full decode starts; each full decode must pass it before it is evaluated.

In [ ]:
for arm in ARMS:
    if not os.path.exists(f"runs/exp-040/{arm}/eval/eval_metrics.json"):
        run_step("canary", arm)
    run_step(arm)

In [ ]:
# ---- conditional arm (pre-declared rule, config.json typed_comparator_rule) ----
if NEED_TYPED_REBUILT:
    run_step("canary", "typed_rebuilt")
    run_step("typed_rebuilt")
else:
    print("typed_rebuilt not needed: typed_rerun's prompts equal typed mode on the rebuilt artifact")

## 3. Paired compare + Holm (CPU, a few minutes)

In [ ]:
run_step("compare")
shutil.copy("experiments/exp-040/compare_metrics.json", f"{DRIVE_OUT}/compare_metrics.json")
m = json.load(open("experiments/exp-040/compare_metrics.json"))
print("typed comparator:", m["typed_comparator"]["system"])
hw = m["hardware_control"]
print(f"hardware control: {hw['identical_decompositions']}/{hw['items']} identical to stored exp-020 ->",
      "reproduced" if hw["reproduced"] else "NOT reproduced")
for fam, f in m["families"].items():
    print(fam, "size", f["size"], "| uncorrected significant", f["uncorrected_significant"], "| Holm significant", f["holm_significant"])

In [ ]:
# ---- what to copy back to the laptop ----
print(f"Everything is in {DRIVE_OUT}/ on Google Drive. Copy back (repo = C:/Users/suuser/thesis/Thesis---QAv2):")
print(f"  {DRIVE_OUT}/runs/exp-040/          -> runs/exp-040/                       (gitignored; whole folder)")
print(f"  {DRIVE_OUT}/compare_metrics.json   -> experiments/exp-040/compare_metrics.json   (committed)")
print(f"  {DRIVE_OUT}/logs/                  -> runs/exp-040/colab_logs/            (logs, environment, GPU)")
print(f"  {DRIVE_OUT}/launch_sha.txt         -> the commit for the log row ({SHA})")
print("Then copy each arm's decomposer config.json into experiments/exp-040/ as snapshots, write notes.md,")
print("and complete the exp-040 row in experiments/log.md (experiments/exp-040/README.md 'After the run').")
for root, _, files in os.walk("runs/exp-040", followlinks=True):
    for f in files:
        if f.endswith((".json", ".jsonl", ".md")) and "prompts_log" not in root:
            print("   ", os.path.join(root, f))